# 🗺️ Chapter 1 — Setting Up: OpenStreetMap as a Data Source
## Python for Urban Analysis — Ghana Edition

---

> **Professor's Welcome:**
> *"Cities are inherently spatial problems — everything comes down to WHERE. This first chapter establishes your environment and the single most important pattern in the course: the study-area pattern. Master this, and every later chapter is just a variation on the same theme."*

### Learning Objectives
1. Understand what OpenStreetMap (OSM) is and why it matters for Africa
2. Install and verify the Python geospatial stack
3. Define a study area and pull your first boundary polygon
4. Establish the "study-area pattern" used in every later chapter

### Key Concepts
- OSM's tag-based data model (nodes → ways → relations)
- Nominatim geocoding (place name → polygon)
- GeoDataFrame basics (geometry + attributes in one table)
- Saving/loading spatial data (GeoPackage — the modern shapefile)

### Ghana Context
Accra is one of the best-mapped cities in Sub-Saharan Africa on OSM, thanks to HOT (Humanitarian OpenStreetMap Team) mapathons and local mapping communities. We use Accra as our running case study — but you can swap it for Kumasi, Tamale, Cape Coast, or any city on Earth.

## 1.0 — Install Dependencies (Google Colab)

> ⚠️ Run this cell **first** every time you open the notebook in Colab.

In [ ]:
# ── Google Colab Setup ──────────────────────────────────────────
!pip install -q osmnx geopandas contextily folium mapclassify momepy networkx

## 1.1 — Verify the Environment

Let's confirm every package we need is installed and check its version.

In [ ]:
import sys
import warnings
warnings.filterwarnings('ignore')

print('=' * 65)
print('  CHAPTER 1 — Environment Check & Study-Area Setup')
print('=' * 65)

packages = {
    'osmnx': None, 'geopandas': None, 'pandas': None,
    'numpy': None, 'matplotlib': None, 'contextily': None,
    'networkx': None, 'shapely': None, 'folium': None,
    'mapclassify': None,
}

for pkg in packages:
    try:
        mod = __import__(pkg)
        packages[pkg] = mod.__version__
        print(f'  ✓ {pkg:20s}  v{mod.__version__}')
    except ImportError:
        print(f'  ✗ {pkg:20s}  — NOT INSTALLED')

missing = [k for k, v in packages.items() if v is None]
if missing:
    print(f'\n⚠  Missing packages: {", ".join(missing)}')
    print('   Run the install cell above!')
else:
    print('\n✅ All packages installed successfully!')

## 1.2 — Configuration

This is the **central configuration** for every chapter. The key idea:

> **Change `STUDY_AREA` to point the entire pipeline at a different city.**

| Variable | Purpose |
|----------|--------|
| `STUDY_AREA` | City-level analysis target |
| `STUDY_REGION` | Region-level target (Ch 6) |
| `CRS_UTM_ACCRA` | Metre-based CRS for area/distance calculations |
| `CRS_WEB_MERCATOR` | Required for contextily basemap tiles |

In [ ]:
import os
from pathlib import Path

# ── Study area (change this to re-target the whole pipeline) ───
STUDY_AREA = 'Accra, Ghana'
STUDY_REGION = 'Greater Accra Region, Ghana'

# ── Coordinate Reference Systems ──────────────────────────────
CRS_WGS84        = 'EPSG:4326'   # Lat/lon — storage & OSM default
CRS_WEB_MERCATOR = 'EPSG:3857'   # Web tiles (contextily)
CRS_UTM_ACCRA    = 'EPSG:32630'  # UTM Zone 30N — metre-based for Accra

# ── Project paths (works in Colab and locally) ────────────────
DATA_DIR    = Path('data')
OUTPUT_DIR  = Path('outputs')
FIGURES_DIR = OUTPUT_DIR / 'figures'

for d in [DATA_DIR, OUTPUT_DIR, FIGURES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ── Visualisation defaults ────────────────────────────────────
DARK_BG      = '#0e1117'
ACCENT_CMAP  = 'YlOrRd'
AMENITY_COLOR = '#00e5ff'

print(f'📍 Study area  → {STUDY_AREA}')
print(f'📂 Data dir    → {DATA_DIR}')
print(f'📂 Output dir  → {OUTPUT_DIR}')

## 1.3 — The Study-Area Pattern

This is the **single most important pattern** in the entire course:

```
1. Define a place name string    →   "Accra, Ghana"
2. Nominatim resolves it         →   administrative polygon
3. Every chapter clips to it     →   consistent spatial extent
```

**Why this matters:**
- **Reproducibility** — anyone can re-run with a different city name
- **Efficiency** — you only download data you need
- **Comparability** — same boundary = same denominator for all metrics

### How OSM Works

OpenStreetMap stores geographic data using three primitives:
- **Nodes** — single points (lat, lon)
- **Ways** — ordered lists of nodes (roads, building outlines)
- **Relations** — groups of ways/nodes (admin boundaries, multipolygons)

Each element carries **tags** — key=value pairs like `amenity=hospital` or `building=residential`. This tag system is how we query OSM for specific urban features.

In [ ]:
import osmnx as ox
import geopandas as gpd

# ── ONE LINE to get a city boundary from anywhere on Earth ────
study_area_gdf = ox.geocode_to_gdf(STUDY_AREA)

print(f'📍 Study area: {STUDY_AREA}')
print(f'   CRS           : {study_area_gdf.crs}')
print(f'   Geometry type : {study_area_gdf.geometry.iloc[0].geom_type}')
print(f'   Bounding box  : {study_area_gdf.total_bounds}')

## 1.4 — Explore the Boundary GeoDataFrame

A **GeoDataFrame** is a pandas DataFrame with a special `geometry` column. It's the backbone of geospatial Python — every spatial dataset you work with will be a GeoDataFrame.

> **Professor's note:** *"Always compute area in a metre-based CRS (like UTM), never in lat/lon (EPSG:4326). Degrees are not constant-length units — 1° of longitude shrinks as you move away from the equator."*

In [ ]:
# ── Examine what Nominatim returned ──────────────────────────
print('── Boundary attributes ──')
print(study_area_gdf[['display_name', 'type', 'geometry']].to_string())

# ── Calculate area in sq-km using a metre-based CRS ──────────
area_m2 = study_area_gdf.to_crs(CRS_UTM_ACCRA).area.iloc[0]
area_km2 = area_m2 / 1e6
print(f'\n   Area ≈ {area_km2:.2f} km²')

## 1.5 — Save to GeoPackage

**GeoPackage** (`.gpkg`) is the modern replacement for shapefiles:
- Single file (no `.shx`, `.dbf`, `.prj` sidecars)
- Supports long column names
- Stores CRS metadata properly
- Open standard (OGC)

In [ ]:
boundary_path = DATA_DIR / 'study_area_boundary.gpkg'
study_area_gdf.to_file(boundary_path, driver='GPKG')
print(f'💾 Boundary saved → {boundary_path}')

## 1.6 — Visualise: Boundary on a Dark Basemap

We use **contextily** to add web map tiles behind our data. The `CartoDB.DarkMatter` tiles give us that sleek, professional dark background that makes data layers pop.

> **Important:** Contextily requires data in **EPSG:3857** (Web Mercator) — the same projection used by Google Maps, OpenStreetMap, and all web tile providers.

In [ ]:
import matplotlib.pyplot as plt
import contextily as cx

fig, ax = plt.subplots(1, 1, figsize=(12, 12))
fig.patch.set_facecolor(DARK_BG)
ax.set_facecolor(DARK_BG)

# Plot boundary — semi-transparent fill + bright outline
study_area_gdf.to_crs(CRS_WEB_MERCATOR).plot(
    ax=ax,
    facecolor='none',
    edgecolor='#ffffff',
    linewidth=2.5,
)

# Add dark basemap tiles
cx.add_basemap(ax, source=cx.providers.CartoDB.DarkMatter)

ax.set_axis_off()
ax.set_title(
    f'Study Area — {STUDY_AREA}',
    fontsize=16, fontweight='bold', color='white', pad=20
)

fig.tight_layout()
fig.savefig(FIGURES_DIR / 'ch1_study_area.png', dpi=150, bbox_inches='tight', facecolor=DARK_BG)
plt.show()

## 1.7 — Interactive Map with Folium

Static maps are great for publications; **interactive maps** are great for exploration. Folium wraps Leaflet.js so you can pan, zoom, and click.

In [ ]:
import folium

centroid = study_area_gdf.geometry.iloc[0].centroid
m = folium.Map(
    location=[centroid.y, centroid.x],
    zoom_start=12,
    tiles='CartoDB dark_matter',
)

# Add boundary as GeoJSON overlay
folium.GeoJson(
    study_area_gdf.to_json(),
    style_function=lambda x: {
        'fillColor': 'transparent',
        'color': '#00e5ff',
        'weight': 3,
    },
    tooltip=STUDY_AREA,
).add_to(m)

m.save(str(OUTPUT_DIR / 'ch1_study_area_interactive.html'))
m  # Display in notebook

## ✅ Chapter 1 Complete

### What you accomplished:
- Verified the Python geospatial stack
- Pulled the Accra boundary from OSM using one line of code
- Saved it as a GeoPackage
- Visualised it on a dark basemap (static + interactive)

### Key takeaway:
> The **study-area pattern** — define a place name, resolve to polygon, clip everything to it — is the foundation for every chapter that follows.

### Next → Chapter 2: Acquiring Urban Data from OSM
We'll pull hospitals, schools, markets, parks, and more.